In [4]:
!pip install groq scikit-learn

In [5]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")

In [8]:
# =============================================================================
# NOTEBOOK 5 — End-to-End Sequential Pipeline
# =============================================================================
# STAGE 1: TRAIN     -- spec_only CRNN detector (or load an existing checkpoint)
# STAGE 2: EVALUATE   -- run on the real test set, save per-sample predictions
# STAGE 3: SELECT     -- auto-pick a few interesting cases (high-confidence
#                        abnormal, borderline, confident normal) from the
#                        real results -- not hand-picked examples
# STAGE 4: EXPLAIN    -- feed each selected case through the RAG + agent +
#                        LLM diagnostic layer, grounded in this project's
#                        own documented findings
#
# This is one script, run top to bottom, so the connection between "the
# model trained on real data" and "the explanation the agent gives" is
# direct and demonstrable -- not two separate, disconnected artifacts.
#
# SETUP ON KAGGLE
#   1. Attach your precomputed dataset as input.
#   2. GPU accelerator on.
#   3. Internet on (needed for the Groq API call in Stage 4).
#   4. Set GROQ_API_KEY via Kaggle Secrets (see Stage 4 section below).
#   5. pip install groq scikit-learn  (torch/torchaudio already on Kaggle)
# =============================================================================

# %% [0] Config -- toggle quick demo mode vs full training
QUICK_DEMO = True   # True: ~5 epochs, fast, for demoing the pipeline mechanics
                     # False: full 20-epoch training matching your validated runs
SKIP_TRAINING_IF_CHECKPOINT_EXISTS = True   # reuse a prior checkpoint if found,
                                              # instead of retraining from scratch

import os
import glob
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

OUTPUT_DIR = "/kaggle/working/end_to_end_pipeline"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

CONFIG = {
    "hop_len": 160, "n_mels": 128,
    "train_crop_samples": 48000,
    "batch_size": 32, "num_workers": 2,
    "epochs": 5 if QUICK_DEMO else 20,
    "lr": 2.5e-4, "weight_decay": 1e-4,
    "warmup_epochs": 2 if QUICK_DEMO else 8,
    "decay_every": 2, "decay_factor": 0.9,
    "grad_clip": 5.0,
    "embed_dim_spec": 96, "fused_dim": 128,
}
STATUS_TO_IDX = {"normal": 0, "abnormal": 1}


# =============================================================================
# STAGE 1a — Data loading
# =============================================================================
def find_dataset_root(slug_hint="mimii-pump-precomputed"):
    explicit_path = "/kaggle/input/datasets/pradeepp23alr067/mimii-pump-precomputed-features-v2"
    if os.path.exists(explicit_path):
        return explicit_path
    candidates = glob.glob(f"/kaggle/input/*{slug_hint}*")
    if candidates:
        return candidates[0]
    raise FileNotFoundError("Could not find dataset. Check /kaggle/input contents.")


def load_manifest():
    dataset_root = find_dataset_root()
    direct_path = os.path.join(dataset_root, "manifest.csv")
    if os.path.exists(direct_path):
        manifest_path, root = direct_path, dataset_root
    else:
        matches = glob.glob(os.path.join(dataset_root, "**", "manifest.csv"), recursive=True)
        if not matches:
            raise FileNotFoundError(f"manifest.csv not found under {dataset_root}")
        manifest_path, root = matches[0], os.path.dirname(matches[0])

    df = pd.read_csv(manifest_path)
    old_prefix = "/kaggle/working/mimii_pump_precomputed"
    for col in ["path_1d", "path_2d"]:
        df[col] = df[col].str.replace(old_prefix, root, regex=False)
    print(f"[Stage 1] Manifest loaded: {len(df)} rows from {root}")
    return df


def random_crop_spec(log_mel_2d, waveform_1d, crop_samples, hop_len, rng):
    total = waveform_1d.shape[-1]
    if crop_samples >= total:
        return log_mel_2d
    start_sample = int(rng.integers(0, total - crop_samples + 1))
    crop_frames = crop_samples // hop_len
    start_frame = start_sample // hop_len
    return log_mel_2d[..., start_frame:start_frame + crop_frames]


class SpecDataset(Dataset):
    def __init__(self, df, split, config, train):
        self.rows = df[df["split"] == split].reset_index(drop=True)
        self.config = config
        self.train = train
        self.rng = np.random.default_rng(SEED)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows.iloc[i]
        waveform_1d = np.load(row["path_1d"])
        log_mel_2d = np.load(row["path_2d"])
        if self.train:
            spec = random_crop_spec(log_mel_2d, waveform_1d, self.config["train_crop_samples"],
                                     self.config["hop_len"], self.rng)
            target_frames = self.config["train_crop_samples"] // self.config["hop_len"]
            if spec.shape[-1] < target_frames:
                spec = np.pad(spec, ((0, 0), (0, 0), (0, target_frames - spec.shape[-1])), mode="edge")
            elif spec.shape[-1] > target_frames:
                spec = spec[..., :target_frames]
        else:
            spec = log_mel_2d
        status = STATUS_TO_IDX[row["status"]]
        return {
            "spec": torch.from_numpy(np.ascontiguousarray(spec.astype(np.float32))),
            "status": torch.tensor(status, dtype=torch.long),
            "snr": row["snr"], "machine_id": row["machine_id"],
        }


def eval_collate(batch):
    max_len = max(b["spec"].shape[-1] for b in batch)
    specs = []
    for b in batch:
        s = b["spec"]
        if s.shape[-1] < max_len:
            s = F.pad(s, (0, max_len - s.shape[-1]), mode="replicate")
        specs.append(s)
    return {
        "spec": torch.stack(specs),
        "status": torch.stack([b["status"] for b in batch]),
        "snr": [b["snr"] for b in batch], "machine_id": [b["machine_id"] for b in batch],
    }


# =============================================================================
# STAGE 1b — Model (spec_only CRNN, your validated best architecture)
# =============================================================================
class CRNNEncoder(nn.Module):
    def __init__(self, embed_dim=96, n_mels=128, gru_hidden=64, gru_layers=1):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d((2, 2)),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d((2, 2)),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d((2, 2)),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d((2, 1)),
        )
        freq_after = n_mels // 16
        self.gru = nn.GRU(64 * freq_after, gru_hidden, gru_layers, batch_first=True, bidirectional=True)
        self.proj = nn.Linear(gru_hidden * 2, embed_dim)

    def forward(self, x):
        x = self.cnn(x)
        b, c, f, t = x.shape
        x = x.permute(0, 3, 1, 2).reshape(b, t, c * f)
        out, _ = self.gru(x)
        return self.proj(out.mean(dim=1))


class SpecOnlyModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.encoder = CRNNEncoder(embed_dim=config["embed_dim_spec"], n_mels=config["n_mels"])
        self.head = nn.Sequential(
            nn.Linear(config["embed_dim_spec"], config["fused_dim"]),
            nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(config["fused_dim"], 2),
        )

    def forward(self, spec):
        return self.head(self.encoder(spec))


# =============================================================================
# STAGE 1c — Training loop
# =============================================================================
def train_model(manifest_df, config):
    checkpoint_path = os.path.join(OUTPUT_DIR, "spec_only_checkpoint.pt")

    train_ds = SpecDataset(manifest_df, "train", config, train=True)
    val_ds = SpecDataset(manifest_df, "val", config, train=False)

    if SKIP_TRAINING_IF_CHECKPOINT_EXISTS and os.path.exists(checkpoint_path):
        print(f"\n[Stage 1] Found existing checkpoint at {checkpoint_path} -- loading instead of retraining.")
        model = SpecOnlyModel(config).to(DEVICE)
        model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
        return model

    print(f"\n[Stage 1] TRAINING -- spec_only CRNN, {config['epochs']} epochs "
          f"({'quick demo' if QUICK_DEMO else 'full'} mode)")

    train_status = train_ds.rows["status"].map(STATUS_TO_IDX).values
    class_counts = np.bincount(train_status)
    sample_weights = (1.0 / class_counts)[train_status]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], sampler=sampler,
                               num_workers=config["num_workers"], drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False,
                             num_workers=config["num_workers"], collate_fn=eval_collate)

    model = SpecOnlyModel(config).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])

    def lr_lambda(epoch):
        if epoch < config["warmup_epochs"]:
            return (epoch + 1) / config["warmup_epochs"]
        return config["decay_factor"] ** ((epoch - config["warmup_epochs"]) // config["decay_every"])

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
    criterion = nn.CrossEntropyLoss()

    def run_epoch(loader, train):
        model.train() if train else model.eval()
        probs_all, labels_all = [], []
        with torch.set_grad_enabled(train):
            for batch in loader:
                spec, status = batch["spec"].to(DEVICE), batch["status"].to(DEVICE)
                with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                    logits = model(spec)
                    loss = criterion(logits, status)
                if train:
                    optimizer.zero_grad()
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
                    scaler.step(optimizer)
                    scaler.update()
                probs = F.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
                probs_all.extend(probs.tolist())
                labels_all.extend(status.detach().cpu().numpy().tolist())
        auc = roc_auc_score(labels_all, probs_all) if len(set(labels_all)) > 1 else float("nan")
        return auc

    best_auc, best_state = -1, None
    for epoch in range(config["epochs"]):
        train_auc = run_epoch(train_loader, train=True)
        val_auc = run_epoch(val_loader, train=False)
        scheduler.step()
        print(f"  Epoch {epoch+1}/{config['epochs']} | train_auc={train_auc:.4f} val_auc={val_auc:.4f}")
        if not np.isnan(val_auc) and val_auc > best_auc:
            best_auc, best_state = val_auc, {k: v.clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    torch.save(model.state_dict(), checkpoint_path)
    print(f"[Stage 1] Training complete. Best val AUC: {best_auc:.4f}. Checkpoint saved.")
    return model


# =============================================================================
# STAGE 2 — Evaluate on the real test set, save per-sample predictions
# =============================================================================
def evaluate_model(model, manifest_df, config):
    print("\n[Stage 2] EVALUATING on test set...")
    test_ds = SpecDataset(manifest_df, "test", config, train=False)
    test_loader = DataLoader(test_ds, batch_size=config["batch_size"], shuffle=False,
                              num_workers=config["num_workers"], collate_fn=eval_collate)

    model.eval()
    records = []
    with torch.no_grad():
        for batch in test_loader:
            spec = batch["spec"].to(DEVICE)
            status = batch["status"].numpy()
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                logits = model(spec)
            probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
            for i in range(len(status)):
                records.append({
                    "snr": batch["snr"][i], "machine_id": batch["machine_id"][i],
                    "label": int(status[i]), "prob": float(probs[i]),
                })

    results_df = pd.DataFrame(records)
    results_df["pred"] = (results_df["prob"] >= 0.5).astype(int)

    overall_auc = roc_auc_score(results_df["label"], results_df["prob"])
    overall_acc = accuracy_score(results_df["label"], results_df["pred"])
    print(f"[Stage 2] Test set: overall AUC={overall_auc:.4f}, ACC={overall_acc:.4f}, n={len(results_df)}")

    csv_path = os.path.join(OUTPUT_DIR, "test_predictions.csv")
    results_df.to_csv(csv_path, index=False)
    print(f"[Stage 2] Per-sample predictions saved to {csv_path}")
    return results_df


# =============================================================================
# STAGE 3 — Auto-select interesting cases from the REAL results (not hand-picked)
# =============================================================================
def select_cases_for_explanation(results_df):
    print("\n[Stage 3] SELECTING representative cases from real test predictions...")
    cases = []

    high_conf_abnormal = results_df[results_df["label"] == 1].sort_values("prob", ascending=False)
    if len(high_conf_abnormal) > 0:
        cases.append(("Highest-confidence true abnormal detection", high_conf_abnormal.iloc[0]))

    borderline = results_df.iloc[(results_df["prob"] - 0.5).abs().argsort()[:1]]
    if len(borderline) > 0:
        cases.append(("Most borderline / uncertain prediction", borderline.iloc[0]))

    confident_normal = results_df[results_df["label"] == 0].sort_values("prob")
    if len(confident_normal) > 0:
        cases.append(("Highest-confidence true normal detection", confident_normal.iloc[0]))

    for label, row in cases:
        print(f"  - {label}: {row['machine_id']} / {row['snr']} / prob={row['prob']:.3f}")
    return cases


# =============================================================================
# STAGE 4 — Diagnostic agent: RAG + rule-based decision + LLM explanation
# (same logic as diagnostic_agent.py, embedded here so this is one self-
# contained runnable pipeline)
# =============================================================================
KNOWLEDGE_BASE = [
    {"id": "fault_types_general", "applies_to": "all", "text": (
        "The MIMII pump dataset documents four qualitative fault types across pump units: "
        "cavitation (fluid vaporization causing internal bubble implosions and structural erosion), "
        "bearing defects (frictional degradation, race cracking, structural clicking), impeller "
        "imbalance (rotational asymmetry causing harmonic vibration), and system leakage (fluid "
        "bypass, pressure loss). These are documented failure categories in the literature, not "
        "per-sample ground-truth labels in this dataset -- individual recordings are labelled only "
        "as normal or abnormal."
    )},
    {"id": "pump06_reliability", "applies_to": ["id_06"], "text": (
        "Pump ID 06 shows consistently reduced detection reliability at -6dB (heavy background "
        "noise) across every model configuration tested in this project, including the base paper's "
        "own Tiny-AST model (AUC 0.836). This project's own spec_only CRNN model reached AUC 0.944 "
        "(pooled training) and 0.884 (isolated per-pump training) on this specific pump/noise "
        "combination -- still the lowest-confidence cell in the full results table. Manual "
        "verification is recommended for Pump 06 detections under heavy noise conditions specifically."
    )},
    {"id": "snr_general_reliability", "applies_to": "all", "text": (
        "Detection reliability across all tested models degrades as SNR decreases. At +6dB and 0dB "
        "(low-to-moderate noise), all tested models in this project achieved AUC above 0.97, "
        "effectively near-ceiling performance. At -6dB (heavy noise), performance drops meaningfully "
        "and becomes pump-dependent, with Pump 06 as the weakest case and Pump 02 typically the "
        "most robust."
    )},
    {"id": "model_choice_rationale", "applies_to": "all", "text": (
        "This project's validated primary detection model is a lightweight CRNN (convolutional + "
        "bidirectional GRU) operating on log-Mel spectrograms, 308K parameters, which outperformed "
        "a faithfully reproduced Tiny-AST transformer baseline (800K-1.01M parameters) on overall "
        "accuracy and noise robustness under identical evaluation protocol."
    )},
    {"id": "domain_adaptation_finding", "applies_to": "all", "text": (
        "Domain-adversarial adaptation techniques were tested to improve robustness at -6dB, but "
        "were found to underperform plain supervised training overall in this project's controlled "
        "multi-seed study. One narrow exception: zero-shot domain generalization matched baseline "
        "performance specifically on Pump 06 at -6dB without ever training on real -6dB audio."
    )},
]

SNR_TO_TEXT = {"pump_6db": "+6dB low noise", "pump_0db": "0dB moderate noise", "pump_minus_6db": "-6dB heavy noise"}
MACHINE_TO_TEXT = {"id_00": "Pump 00", "id_02": "Pump 02", "id_04": "Pump 04", "id_06": "Pump 06"}


def build_retriever(kb):
    texts = [e["text"] for e in kb]
    vec = TfidfVectorizer(stop_words="english")
    return vec, vec.fit_transform(texts)


def retrieve_context(query, kb, vectorizer, matrix, top_k, machine_id):
    """Hybrid retrieval: filter by metadata relevance FIRST, then rank the
    remaining candidates by TF-IDF similarity. Pure TF-IDF over 5 short,
    vocabulary-overlapping entries was returning the same top-2 regardless
    of query -- e.g. pump06_reliability was surfacing even for Pump 00/02/04
    queries, forcing the LLM into an awkward, irrelevant tangent (this was
    the direct cause of the truncated Pump-00 report: it spent its token
    budget trying to connect an irrelevant Pump06 note instead of just
    summarizing the actual case)."""
    eligible = [
        (e, i) for i, e in enumerate(kb)
        if e["applies_to"] == "all" or machine_id in e["applies_to"]
    ]
    if not eligible:
        return []

    indices = [i for _, i in eligible]
    sub_matrix = matrix[indices]
    scores = cosine_similarity(vectorizer.transform([query]), sub_matrix)[0]
    ranked = sorted(zip([e for e, _ in eligible], scores), key=lambda x: x[1], reverse=True)
    return [e for e, s in ranked[:top_k] if s > 0]


def agent_decide(probability, snr, machine_id):
    if probability >= 0.85:
        action, priority, note = "auto-flag for maintenance", "high", "High-confidence detection."
    elif probability >= 0.6:
        action, priority, note = "flag for maintenance, recommend verification", "medium", \
            "Moderate confidence -- within the range where models show more disagreement."
    elif probability >= 0.4:
        action, priority, note = "escalate for manual inspection, do not auto-act", "review-required", \
            "Borderline confidence -- automated action not recommended at this probability range."
    else:
        action, priority, note = "no action", "low", "Low abnormal probability."

    if machine_id == "id_06" and snr == "pump_minus_6db" and priority == "high":
        priority = "high, manual verification recommended"
        note += " Note: this pump/noise combination is this project's documented weakest detection case."
    return action, priority, note


def generate_report(client, model_name, probability, machine_id, snr, action, priority, note, context_entries):
    if context_entries:
        context_text = "\n\n".join(f"[{e['id']}]: {e['text']}" for e in context_entries)
    else:
        context_text = "(No specifically relevant documented findings for this pump/condition combination.)"

    prompt = f"""You are a maintenance-reporting assistant for an industrial pump monitoring system.

A trained fault-detection model has produced the following result:
- Machine: {machine_id}
- Noise condition (SNR): {snr}
- Abnormal probability: {probability:.2f}

The monitoring system's decision logic has already determined:
- Recommended action: {action}
- Priority: {priority}
- Decision note: {note}

Retrieved context from this project's documented findings (use ONLY this context,
do not introduce outside claims or invent fault diagnoses not supported here.
If the context says there are no specifically relevant findings, do not force
an irrelevant connection -- just state the result plainly):
{context_text}

Write the report using EXACTLY this structure, one short line per field
(this is a strict template -- do not add extra sections, do not use numbered
lists or bold headers beyond what is shown):

Detection Summary: <one sentence: machine, condition, probability>
Decision: <action> | Priority: <priority>
Reliability Context: <ALWAYS state the general detection reliability level for
this specific noise condition first, based on the retrieved context -- e.g.
"near-ceiling reliability (AUC > 0.97) at this noise level" for +6dB/0dB, or
"reduced and pump-dependent reliability" for -6dB. Then, ONLY IF the retrieved
context also contains a caveat specific to this exact pump, append it as a
second short sentence. Never state "no concerns documented" if a general
SNR-level reliability fact is available in the retrieved context -- that fact
itself IS the reliability context.>
Recommended Next Step: <one sentence, concrete action for the engineer>

Keep the ENTIRE response under 100 words total. Do not claim a specific fault
type unless the context explicitly supports it for this exact case."""

    response = client.chat.completions.create(
        model=model_name,
        max_tokens=600,          # gpt-oss-120b spends tokens on internal reasoning
                                   # BEFORE writing the visible answer -- too low a
                                   # budget (e.g. 250) can consume it all on reasoning
                                   # and return an EMPTY response with no error. 600
                                   # leaves comfortable room for both.
        reasoning_effort="low",   # this is a templated report task, not a task that
                                   # benefits from deep reasoning -- low effort leaves
                                   # more of the token budget for the actual answer
        messages=[{"role": "user", "content": prompt}],
    )
    content = response.choices[0].message.content
    if not content or not content.strip():
        # defensive fallback: if it's still empty for any reason, surface that
        # clearly instead of silently printing a blank explanation
        return (f"[No response generated -- finish_reason="
                f"{response.choices[0].finish_reason}. Consider raising max_tokens further.]")
    return content


def run_explanation_stage(cases):
    print("\n[Stage 4] EXPLAINING selected cases via RAG + agent + LLM...")

    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        print("  GROQ_API_KEY not set -- skipping LLM explanation step.")
        print("  (Set it via Kaggle Secrets; see the setup notes at the top of this script.)")
        return []

    from groq import Groq
    client = Groq(api_key=api_key)
    model_name = "openai/gpt-oss-120b"
    vectorizer, matrix = build_retriever(KNOWLEDGE_BASE)

    all_reports = []
    for label, row in cases:
        pump_text = MACHINE_TO_TEXT.get(row["machine_id"], row["machine_id"])
        snr_text = SNR_TO_TEXT.get(row["snr"], row["snr"])
        query = f"{pump_text} {snr_text} fault detection reliability abnormal probability {row['prob']}"
        context_entries = retrieve_context(query, KNOWLEDGE_BASE, vectorizer, matrix, top_k=2,
                                            machine_id=row["machine_id"])
        action, priority, note = agent_decide(row["prob"], row["snr"], row["machine_id"])
        report = generate_report(client, model_name, row["prob"], row["machine_id"], row["snr"],
                                  action, priority, note, context_entries)

        print("=" * 80)
        print(f"Case: {label}")
        print(f"Machine: {row['machine_id']} | SNR: {row['snr']} | True label: {row['label']} | "
              f"Model probability: {row['prob']:.3f}")
        print(f"Agent decision: {action} (priority: {priority})")
        print(f"Retrieved context: {[e['id'] for e in context_entries]}")
        print(f"\nExplanation:\n{report}\n")

        all_reports.append({
            "case_label": label, "machine_id": row["machine_id"], "snr": row["snr"],
            "true_label": int(row["label"]), "probability": float(row["prob"]),
            "action": action, "priority": priority,
            "retrieved_context_ids": [e["id"] for e in context_entries], "report": report,
        })

    with open(os.path.join(OUTPUT_DIR, "end_to_end_reports.json"), "w") as f:
        json.dump(all_reports, f, indent=2)
    print(f"[Stage 4] All explanations saved to {OUTPUT_DIR}/end_to_end_reports.json")
    return all_reports


# =============================================================================
# MAIN — run the full sequential pipeline
# =============================================================================
if __name__ == "__main__":
    manifest_df = load_manifest()
    model = train_model(manifest_df, CONFIG)
    test_results_df = evaluate_model(model, manifest_df, CONFIG)
    selected_cases = select_cases_for_explanation(test_results_df)
    reports = run_explanation_stage(selected_cases)

    print("\n" + "=" * 80)
    print("PIPELINE COMPLETE: train -> evaluate -> select -> explain")
    print(f"All artifacts in {OUTPUT_DIR}:")
    print("  - spec_only_checkpoint.pt   (trained model)")
    print("  - test_predictions.csv      (real per-sample predictions)")
    print("  - end_to_end_reports.json   (agent explanations for selected cases)")
    print("=" * 80)

Using device: cuda
GPU: Tesla T4
[Stage 1] Manifest loaded: 12615 rows from /kaggle/input/datasets/pradeepp23alr067/mimii-pump-precomputed-features-v2/mimii_pump_precomputed

[Stage 1] Found existing checkpoint at /kaggle/working/end_to_end_pipeline/spec_only_checkpoint.pt -- loading instead of retraining.

[Stage 2] EVALUATING on test set...
[Stage 2] Test set: overall AUC=0.9873, ACC=0.9668, n=1266
[Stage 2] Per-sample predictions saved to /kaggle/working/end_to_end_pipeline/test_predictions.csv

[Stage 3] SELECTING representative cases from real test predictions...
  - Highest-confidence true abnormal detection: id_00 / pump_6db / prob=1.000
  - Most borderline / uncertain prediction: id_00 / pump_0db / prob=0.501
  - Highest-confidence true normal detection: id_04 / pump_minus_6db / prob=0.000

[Stage 4] EXPLAINING selected cases via RAG + agent + LLM...
Case: Highest-confidence true abnormal detection
Machine: id_00 | SNR: pump_6db | True label: 1 | Model probability: 1.000
Agent 